# Training a language model to do maths with GRPO

### AI4Science Summer School — Caen, 2 July 2026
*Pierre Marion & Gaëtan Narozniak*

> **How to run.**
> Use a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`).

*Adapted from the GRPO example by [Maxime Labonne](https://huggingface.co/mlabonne); see the original [notebook](https://huggingface.co/course/en/chapter12/5).*

,Welcome to the hands-on session! During the talk we saw the big picture:

1. Language models are **trained to predict the next token** on huge amounts of text.
2. They are then refined to **learn from our preferences** and to **solve verifiable problems** (maths, code, ...).
3. **GRPO** is one of the key algorithms behind that last step, and it is at the heart of modern *"reasoning"* models.

In this notebook we will:

| Part | What we do |
|------|-----------|
| **1** | See **next-token prediction** in action with GPT-2 |
| **2** | Meet our small chat model and a **toy maths task** (multiplication) |
| **3** | Build the two ingredients of GRPO: a **reward** and a **dataset** |
| **4** | **Train** the model with GRPO and watch it improve |
| **5** | **Experiment**: change the reward and the dataset, and see what happens |


## Setup

We first install and import the libraries. This is standard boilerplate — you do not need to understand it in detail.

- [`transformers`](https://huggingface.co/docs/transformers): load pretrained language models,
- [`datasets`](https://huggingface.co/docs/datasets): handle datasets,
- [`trl`](https://huggingface.co/docs/trl): the *"Transformer Reinforcement Learning"* library, which contains a ready-to-use GRPO trainer,
- [`peft`](https://huggingface.co/docs/peft): train only a small fraction of the weights (LoRA), to make training fast.


In [ ]:
!pip install -qqq datasets transformers trl==1.6.0 peft accelerate bitsandbytes
!pip install -qqq --upgrade torchao

In [ ]:
import re
import random

import torch
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from transformers.utils.notebook import NotebookProgressCallback
from trl import GRPOConfig, GRPOTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Part 1 — Next-token prediction with GPT-2

Everything a language model does is built on a single, surprisingly simple task:

> **Given a piece of text, predict the next *token*.**

A *token* is a chunk of text (a word, a piece of a word, a punctuation mark...). The model reads a sequence of tokens and outputs a **probability distribution over the vocabulary** of possible next tokens.

To *see* this concretely, we load **GPT-2** (2019), the ancestor of today's models. It is small (124M parameters) and is a pure *base* model: it has only been trained to predict the next token on internet text, nothing else.


In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
gpt2_model.eval()

n_params = sum(p.numel() for p in gpt2_model.parameters())
print(f"GPT-2 loaded — {n_params/1e6:.0f} million parameters.")

## 1.1 Tokenization: from text to numbers

A neural network only manipulates numbers, not text. The first step is therefore **tokenization**: the text is cut into tokens, and each token is mapped to an integer id (its index in a vocabulary of size `V`).

Run the cell below and notice a few things:
- common words are a single token, e.g. `" meal"` (the leading space is part of the token!),
- the vocabulary has `V ≈ 50 000` entries for GPT-2.


In [ ]:
text = "My favorite meal is a"

ids = gpt2_tok(text)["input_ids"]

print(f"Vocabulary size V = {gpt2_tok.vocab_size}\n")
print(f"{'token id':>10} | token")
print("-" * 30)
for i in ids:
    print(f"{i:>10} | {gpt2_tok.decode([i])!r}")

## 1.2 One forward pass = a probability over the next token

We now feed the tokens to GPT-2. Internally the model produces a distribution on the most likely next token following the input prefix.


In [ ]:
@torch.no_grad()
def next_token_distribution(prompt, top_k=10):
    inputs = gpt2_tok(prompt, return_tensors="pt").to(device)
    logits = gpt2_model(**inputs).logits        # shape [1, sequence_length, V]
    last_logits = logits[0, -1]                 # logits z for the NEXT token, shape [V]
    probs = torch.softmax(last_logits, dim=-1)  # softmax -> probability distribution
    top = torch.topk(probs, top_k)
    tokens = [gpt2_tok.decode([i]) for i in top.indices]
    return tokens, top.values.cpu().numpy()


prompt = "My favorite meal is a"
tokens, probs = next_token_distribution(prompt)

plt.figure(figsize=(7, 4))
plt.barh(range(len(tokens)), probs, color="#4C9F70")
plt.yticks(range(len(tokens)), [repr(t) for t in tokens])
plt.gca().invert_yaxis()
plt.xlabel("probability of being the next token")
plt.title(f"GPT-2 prediction after:\n{prompt!r}")
plt.tight_layout()
plt.show()

print("Most likely next token:", repr(tokens[0]))

**Try it yourself.** Change the `prompt` in the cell above and re-run it. A few ideas:
- `"Paris is the capital of"` → the model *knows* geography (factual knowledge),
- `"1 2 3 4"` → it can continue simple patterns,
- `"Mr Smith is a good friend of mine. Mr"` → it copies from context.

All of these skills emerge from the *single* objective of predicting the next token.


## 1.3 Generating text = predicting one token at a time

To generate a full answer, the model simply repeats the operation: it predicts the next token, **appends it to the text**, and predicts again. This is called *autoregressive* generation.

The cell below does this by hand, always picking the **most likely** token (this is called *greedy* decoding). Watch the sentence grow one token at a time, together with the probability the model assigned to each choice.


In [ ]:
@torch.no_grad()
def greedy_generate(prompt, n_steps=8):
    ids = gpt2_tok(prompt, return_tensors="pt").to(device)["input_ids"]
    print(f"start: {gpt2_tok.decode(ids[0])!r}\n")
    for _ in range(n_steps):
        logits = gpt2_model(ids).logits
        probs = torch.softmax(logits[0, -1], dim=-1)
        next_id = probs.argmax()
        print(f"  next token = {gpt2_tok.decode([next_id])!r:>14}   (probability {probs[next_id]:.2f})")
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    print(f"\nfull text: {gpt2_tok.decode(ids[0])!r}")


greedy_generate("My favorite meal is a", n_steps=8)

## 1.4 Temperature: being creative vs. being safe

Always taking the most likely token makes the model repetitive. Instead we usually **sample** the next token from the distribution. The **temperature** `T` reshapes the distribution before sampling:
$$p_k \propto e^{z_k / T}.$$

- small `T` puts more weight on likely tokens.
- large `T` yields a flatter distribution, more surprising / creative.

We will use sampling with a temperature later when generating candidate answers for GRPO.


In [ ]:
@torch.no_grad()
def logits_after(prompt):
    inputs = gpt2_tok(prompt, return_tensors="pt").to(device)
    return gpt2_model(**inputs).logits[0, -1]


prompt = "My favorite meal is a"
last_logits = logits_after(prompt)

THRESHOLD = 0.01          # a token is "plausible" if its probability exceeds this

print(f"Prompt: {prompt!r}")
print(f"Plausible next tokens (probability > {THRESHOLD:.0%}), for a few temperatures:")

for T in [0.1, 0.2, 0.8]:
    probs = torch.softmax(last_logits / T, dim=-1)
    idx = torch.where(probs > THRESHOLD)[0]
    idx = idx[torch.argsort(probs[idx], descending=True)]
    print(f"\n  T = {T}   ({len(idx)} plausible tokens)")
    for i in idx:
        p = probs[i].item()
        bar = "█" * max(1, round(p * 40))
        print(f"      {gpt2_tok.decode([i])!r:<14} {p:6.1%}  {bar}")

### ✅ Take-away from Part 1

A language model is a machine that, given some text, outputs a **probability distribution over the next token**. Generation is just repeating this step. Training such a model on the whole internet already teaches it grammar, facts, translation, a bit of reasoning...

But a raw base model like GPT-2 is not good at reasoning/maths. That is what the next parts are about.


# Part 2 — A small chat model and a maths task

Modern assistants start from a base model and are further trained to **follow instructions**, to **match human preferences**, and to **solve verifiable problems**. We will now take a small LLM and improve it on a concrete, verifiable task.

**Our model: `Qwen3-0.6B`.** A small (0.6-billion-parameter) instruction-tuned model. Small enough to train in a few minutes on a free GPU, but already able to *chat* and to *attempt* maths.

**Our task: multiply two integers.** For example *"What is 6 × 8?"*. This task is perfect for GRPO because it is **verifiable**: there is exactly one correct answer and we can check it automatically.


## 2.1 Load the model

We load Qwen3-0.6B.


In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

# The tokenizer is shared by all our experiments, load it once.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"          # required for correct batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_model():
    "Load a fresh Qwen3-0.6B and attach a new (untrained) LoRA adapter."
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
        attn_implementation="sdpa",
    )
    # LoRA (Low-Rank Adaptation) trains only a small fraction of the parameters.
    # This is necessary to speed up training.
    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
    )
    model = get_peft_model(model, lora_config)
    return model


model = build_model()
model.print_trainable_parameters()

## 2.2 Talking to the model

The helper `generate(prompts)` sends a batch of prompts to the model and returns its answers. We will reuse it throughout the notebook.

We ask the model to write its final result between `<answer>` and `</answer>` tags. This gives us an easy, unambiguous way to **read off the answer** and check whether it is correct.


In [ ]:
ANSWER_INSTRUCTION = (
    "Give your final answer between <answer> and </answer>, "
    "for example <answer>132</answer>."
)


def make_prompt(n1, n2):
    return f"What is the value of {n1} * {n2}? {ANSWER_INSTRUCTION}"


@torch.no_grad()
def generate(prompts, max_new_tokens=96, do_sample=False, temperature=0.7):
    "Generate an answer for each prompt (greedy by default)."
    model.eval()
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    # stop_strings makes generation return early as soon as the model closes the
    # answer tag: without it, this model rarely emits an end-of-sequence token and
    # would keep writing until max_new_tokens (slow).
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        stop_strings="</answer>",
        tokenizer=tokenizer,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    output_ids = model.generate(**inputs, **gen_kwargs)
    gen_only = output_ids[:, inputs["input_ids"].shape[1]:]   # keep only the new tokens
    return tokenizer.batch_decode(gen_only, skip_special_tokens=True)


# Try the untrained model on a few prompts
demo_prompts = [
    make_prompt(3, 8),
    make_prompt(7, 9),
    "What is the capital of France? Answer in one word.",
]
for p, answer in zip(demo_prompts, generate(demo_prompts)):
    print("PROMPT :", p)
    print("ANSWER :", answer.strip()[:400])
    print("-" * 80)

Take a moment to read the answers and to test other prompts. Try varying the format instructions and see if the model complies. You will probably see that the model:
- can chat and often *knows* how to multiply small numbers,
- but is **unreliable**: it rambles, sometimes forgets the `<answer>` tags, or makes arithmetic mistakes, especially for larger numbers.

Our goal with GRPO today is to make it **reliably** produce the correct answer in the right format.


## 2.3 Reading the answer and measuring accuracy

To *evaluate* the model we need to (1) extract the number inside the `<answer>` tags and (2) compare it to the ground truth. The same extraction function will be reused by the reward in Part 3.


In [ ]:
def extract_answer(text):
    "Return the content of the FIRST <answer>...</answer>, or None if absent."
    matches = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL)
    return matches[0].strip() if matches else None


def answer_present(text, ground_truth):
    "True if the correct number appears ANYWHERE in the text (not only in the tags)."
    return re.search(rf"(?<!\d){re.escape(ground_truth)}(?!\d)", text) is not None


@torch.no_grad()
def evaluate(dataset, max_new_tokens=96, batch_size=32):
    "Return two metrics on a dataset:"
    "  - accuracy: correct answer AND inside <answer> tags,"
    "  - present : correct answer appears somewhere in the text (format ignored)."
    prompts = [ex["prompt"] for ex in dataset]
    answers = [ex["answer"] for ex in dataset]
    texts = []
    for start in range(0, len(prompts), batch_size):
        texts.extend(generate(prompts[start:start + batch_size], max_new_tokens=max_new_tokens))
    accuracy = sum(extract_answer(t) == a for t, a in zip(texts, answers)) / len(answers)
    present = sum(answer_present(t, a) for t, a in zip(texts, answers)) / len(answers)
    return accuracy, present

# Part 3 — The two ingredients of GRPO

GRPO (*Group Relative Policy Optimization*) is a **reinforcement learning** algorithm. Recall from the talk: in reinforcement learning, an agent takes actions and receives **rewards**; it learns to act so as to collect more reward. Here:

- the **action** = the text the model generates,
- the **reward** = a score saying how good that text is (here: is the answer correct and well-formatted?).

To run GRPO we only need to provide two things:

1. a **reward function** — how we score an answer *(Section 3.1)*,
2. a **dataset** of prompts to practice on *(Section 3.2)*.

Then GRPO does the rest *(Section 3.3)*. These are exactly the two things you will experiment with in Part 5.


## 3.1 The reward function

For each generated answer, the reward function returns a number. Our reward encodes what we want:

| Situation | Reward |
|-----------|-------:|
| Correct answer, inside `<answer>` tags | `REWARD_CORRECT` |
| Wrong answer, but inside `<answer>` tags | `REWARD_WRONG` |
| No `<answer>` tags at all | `REWARD_NO_TAG` |

The reward is the **only signal** the model gets about what "good" means.


In [ ]:
# --- These three numbers define the reward. ---
REWARD_CORRECT = 1.0 ### To be experimented with in Section 5     # answer is correct and well-formatted
REWARD_WRONG   = 0. ### To be experimented with in Section 5     # answer is well-formatted but wrong
REWARD_NO_TAG  = -0.2 ### To be experimented with in Section 5     # model did not use the <answer> tags


def reward_math(completions, **kwargs):
    "Score each generated completion. `kwargs['answer']` holds the ground truths."
    ground_truths = kwargs["answer"]
    rewards = []
    for completion, ground_truth in zip(completions, ground_truths):
        predicted = extract_answer(completion)
        if predicted is None:
            rewards.append(REWARD_NO_TAG)
        elif predicted == ground_truth:
            rewards.append(REWARD_CORRECT)
        else:
            rewards.append(REWARD_WRONG)
    return rewards


def answer_char_pos(completions, **kwargs):
    "Tracker only: character position of the <answer> tag (len(text) if absent)."
    "Added to the trainer with weight 0, so it is logged but does NOT affect training."
    positions = []
    for c in completions:
        pos = c.find("<answer>")
        positions.append(float(pos) if pos != -1 else float(len(c)))
    return positions


# Quick sanity check on three fake completions
demo_completions = [
    "The result is <answer>42</answer>",   # correct
    "The result is <answer>99</answer>",   # wrong
    "I think it is 42",                     # no tags
]
print(reward_math(demo_completions, answer=["42", "42", "42"]))

## 3.2 The dataset 📚

The dataset is just a list of `{"prompt": ..., "answer": ...}`. We generate multiplication problems automatically.

Two design choices are interesting to play with (Part 5):
- **the range of the factors** (`MIN_FACTOR`, `MAX_FACTOR`): bigger numbers = harder problems,
- **the train/test split**: we hold out a few numbers (`TEST_FACTORS`) that are **never seen during training**. Measuring accuracy on those tells us whether the model *learned to multiply* or merely *memorized* the training pairs.


In [ ]:
# --- Dataset configuration. ---
MIN_FACTOR : int   = 1 ### To be experimented with in Section 5
MAX_FACTOR : int   = 19 ### To be experimented with in Section 5
TEST_FACTORS : list[int] = [4, 7, 12, 15] ### To be experimented with in Section 5     # numbers held out for testing (never seen in training)
TRAIN_SIZE   = 1000
TEST_SIZE    = 100


def build_datasets(seed=0):
    rng = random.Random(seed)
    train_factors = [n for n in range(MIN_FACTOR, MAX_FACTOR + 1) if n not in TEST_FACTORS]

    def sample(n_examples, factors):
        rows = []
        for _ in range(n_examples):
            n1, n2 = rng.choice(factors), rng.choice(factors)
            rows.append({"prompt": make_prompt(n1, n2), "answer": str(n1 * n2)})
        return Dataset.from_list(rows)

    train_set = sample(TRAIN_SIZE, train_factors)
    test_set = sample(TEST_SIZE, TEST_FACTORS)   # only held-out numbers
    return train_set, test_set


train_set, test_set = build_datasets()
print(f"train examples: {len(train_set)}   |   test examples (held-out numbers): {len(test_set)}")
print("\nExample training prompt:")
print(" ", train_set[0]["prompt"])
print("Example answer:")
print("  The value is: <answer>" + train_set[0]["answer"] + "</answer>")

## 3.3 How GRPO turns rewards into learning

Here is the whole idea of GRPO in one picture. For a given question:

1. **Sample a group** of `G` candidate answers from the current model (using temperature > 0, so they differ).
2. **Score** each with the reward function.
3. Compute each answer's **advantage** = how much better it is than the *average of the group*:
$$A_i = \frac{r_i - \text{mean}(r_1,\dots,r_G)}{\text{std}(r_1,\dots,r_G)}.$$
4. **Update the model**: increase the probability of answers with a *positive* advantage (better than average), decrease it for *negative* ones.


# Part 4 — Training with GRPO

We now put everything together and train. First we measure the model's accuracy **before** training, so we have a baseline to compare against.


In [ ]:
acc_before, present_before = evaluate(test_set)
print(f"BEFORE training (held-out test set):")
print(f"  correct & inside <answer> tags : {acc_before:.0%}")
print(f"  correct number present anywhere: {present_before:.0%}")

## 4.1 Launch training

The `GRPOConfig` below collects the training hyper-parameters. The ones worth knowing:
- `num_generations = 8`: the group size `G` (how many answers per question),
- `max_steps = 20`: how many optimization steps (kept small so training takes ~3 min),
- `max_completion_length = 96`: how many tokens the model may write per answer.

A live table prints one sampled completion per step so you can watch the answers evolve.

> ⏳ **This cell takes a few minutes.** Launch it, and observe the example outputs of the model and the evolution of the metrics as it trains.


In [ ]:
class PrintMetricsCallback(TrainerCallback):
    "Print a compact one-line summary at each logging step."
    def on_log(self, args, state, control, logs=None, **kwargs):
        keep = ["reward", "rewards/answer_char_pos/mean", "entropy", "grad_norm"]
        if logs is not None and state.is_world_process_zero:
            shown = " | ".join(
                f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                for k, v in logs.items() if k in keep
            )
            if shown:
                print(f"[step {state.global_step:>2}] {shown}")


training_args = GRPOConfig(
    output_dir="GRPO",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    max_completion_length=96,
    num_generations=8,
    max_steps=20,
    optim="adamw_8bit",
    bf16=True,
    logging_steps=1,
    num_completions_to_print=1,
    log_completions=True,
    report_to="none",
    remove_unused_columns=False,   # keep the "answer" column so the reward can read it
    reward_weights=[1.0, 0.0],     # reward_math counts; answer_char_pos is tracked only
)

model.train()
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_math, answer_char_pos],   # 2nd one has weight 0 (tracking only)
    args=training_args,
    train_dataset=train_set,
)
trainer.add_callback(PrintMetricsCallback())
trainer.remove_callback(NotebookProgressCallback)

trainer.train()

## 4.2 What to look at during training

Watch the printed **`reward`**: on average it should *increase*, meaning the model produces more correct, well-formatted answers over time.


In [ ]:
history = [h for h in trainer.state.log_history if "reward" in h]
steps = [h["step"] for h in history]
rewards = [h["reward"] for h in history]
answer_pos = [h.get("rewards/answer_char_pos/mean", np.nan) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, rewards, marker="o", color="#4C9F70")
ax1.set_title("Mean reward per step")
ax1.set_xlabel("training step")
ax1.set_ylabel("mean reward")
ax1.grid(alpha=0.3)

ax2.plot(steps, answer_pos, marker="o", color="#3B7DD8")
ax2.set_title("Mean first position of <answer>")
ax2.set_xlabel("training step")
ax2.set_ylabel("character index of <answer>")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4.3 Did it work? Before vs. after

We re-measure both metrics on the **held-out** test numbers and compare with the baseline. Remember: these numbers were **never seen during training**, so an improvement means the model got genuinely better, not that it memorized the training pairs.

We plot **two** metrics side by side:
- **correct in `<answer>`**: the answer is right *and* correctly formatted (this is what the reward rewards),
- **answer present anywhere**: the correct number appears somewhere in the text, even outside the tags.

The gap between the two tells you how much of the model's failure is a **formatting** problem vs. a **maths** problem.


In [ ]:
acc_after, present_after = evaluate(test_set)

labels = ["correct in <answer>", "answer present anywhere"]
before = [acc_before, present_before]
after = [acc_after, present_after]

x = np.arange(len(labels))
w = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x - w / 2, before, w, label="before", color="#B0B0B0")
plt.bar(x + w / 2, after, w, label="after", color="#4C9F70")
plt.xticks(x, labels)
plt.ylim(0, 1)
plt.ylabel("fraction of held-out test set")
plt.title("Effect of GRPO training")
plt.legend()
for i, (b, a) in enumerate(zip(before, after)):
    plt.text(i - w / 2, b + 0.02, f"{b:.0%}", ha="center")
    plt.text(i + w / 2, a + 0.02, f"{a:.0%}", ha="center")
plt.tight_layout()
plt.show()

## 4.4 Read the model's answers — and probe it

Let's look at a few full answers. The prompts are written out **in full** below (rather than hidden behind `make_prompt`) so you can edit them directly.

**Try changing them!** Test the model on prompts that are *not* exactly like the training ones, e.g.:
- a different phrasing (*"Compute 12 times 15."*),
- without the worked example in the instruction,
- a different operation (*"What is 13 + 29?"*).

Does the model still wrap its answer in `<answer>...</answer>`? This tells you whether it learned the **format** in general, or only for the exact training prompt.


In [ ]:
prompts = [
    "What is the value of 4 * 18? Give your final answer between <answer> and </answer>, for example <answer>132</answer>.",
    "What is the value of 7 * 13? Give your final answer between <answer> and </answer>, for example <answer>132</answer>.",
    "Compute 12 times 15. Give your final answer between <answer> and </answer>.",
]
for p, answer in zip(prompts, generate(prompts)):
    print("PROMPT   :", p)
    print("ANSWER   :", answer.strip()[:300])
    print("-" * 80)

# Part 5 — Your turn: experiment! 🧪

Change the ingredients and see what happens. For each experiment:

1. edit the values in **Section 3** (the reward and/or the dataset),
2. **re-run every cell from Section 2 onwards** — this reloads a fresh model and retrains (~3 min).

Because each run costs ~3 minutes, **decide what you expect before launching**, then check whether you were right.


## Experiment A — Change the reward (Section 3.1)

Try changing the reward values.

**Think:** what would you expect with `REWARD_CORRECT = -1.0`?

**Think:** why would setting all three rewards equal make learning impossible?

## Experiment B — Change the dataset (Section 3.2)

Can you change the dataset to make the task harder?

**Hint:** think about `MAX_FACTOR` and `TRAIN_SIZE`.

**Think:** the test set uses *held-out* numbers. What does that measure that training accuracy cannot?


## (Bonus) Experiment C — Reward answering early

GRPO can optimize **several rewards at once**: `reward_funcs` takes a *list* of functions, and their scores are combined using `reward_weights`. We already track the character position of `<answer>` with a **weight-0** function (`answer_char_pos`). Now let's turn a similar quantity into an **actual reward** that gives a bonus when the `<answer>` tag appears **early** (at a small token index), to push the model to answer promptly instead of rambling.

**1.** Add this function in **Section 3.1** (next to `reward_math`):

```python
def reward_early_answer(completions, **kwargs):
    "Bonus for placing the <answer> tag at a small token index."
    max_len = 96
    rewards = []
    for c in completions:
        pos = c.find("<answer>")
        if pos == -1:
            rewards.append(0.0)                         # no tag -> no bonus
            continue
        n_tokens = len(tokenizer(c[:pos])["input_ids"])  # tokens before the tag
        rewards.append(0.3 * (1 - min(n_tokens, max_len) / max_len))
    return rewards
```

**2.** In the **training cell (Section 4.1)**, add it to the list *and* give it a non-zero weight (the lists must stay the same length):

```python
reward_funcs=[reward_math, answer_char_pos, reward_early_answer]
...
reward_weights=[1.0, 0.0, 1.0]     # answer_char_pos stays a weight-0 tracker
```

Retrain and watch the **"Mean character position of `<answer>`"** plot (and the `rewards/answer_char_pos/mean` line in the logs): it should now **drop** compared to the baseline run. Is accuracy affected?

**Think:** what happens if you make the bonus too large (e.g. `0.3 -> 2.0`)? The model may rush out an early `<answer>` with a *wrong* number just to grab the bonus — another example of **reward hacking**.

# Wrap-up

You just ran the same kind of training loop that powers modern *reasoning* models — only smaller. The key ideas to remember:

- A language model **predicts the next token**; generation repeats this step *(Part 1)*.
- On **verifiable tasks** we can score answers automatically with a **reward**, no human labels needed *(Part 3)*.
- **GRPO** samples a group of answers, keeps what is **better than average**, and nudges the model towards it *(Part 3.3)*.
- With just a **reward** and a **dataset**, a small model measurably improves — and, crucially, on numbers it **never saw during training** *(Part 4)*.

Thanks for participating! 🎉

